In [40]:
import os
import glob
import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 路径配置 ---
base_dir = r'D:\Project_Github\Indo-Pacific-humpback-dolphin'
data_dir = os.path.join(base_dir, '00_Data', 'ClickTrains')
output_base = os.path.join(base_dir, '04_Distinguish','Output_Classification')

# 创建输出子文件夹
categories = ['Clean', 'Moderate', 'Severe']
for cat in categories:
    os.makedirs(os.path.join(output_base, cat), exist_ok=True)

print(f"数据目录: {data_dir}")
print(f"输出目录: {output_base}")

数据目录: D:\Project_Github\Indo-Pacific-humpback-dolphin\00_Data\ClickTrains
输出目录: D:\Project_Github\Indo-Pacific-humpback-dolphin\04_Distinguish\Output_Classification


In [41]:
def calculate_teager_energy(signal):
    """计算 Teager 能量算子"""
    # x[n]^2 - x[n-1]*x[n+1]
    teo = signal[1:-1]**2 - signal[:-2] * signal[2:]
    # 补齐首尾，保持长度一致
    return np.pad(teo, (1, 1), mode='edge')

def moving_average(data, window_size=10):
    """滑动平均"""
    return np.convolve(data, np.ones(window_size)/window_size, mode='same')

def detect_boundaries(signal, fs):
    """
    根据 Teager 能量和阈值检测信号边界
    修改：从能量最大值点开始向前后搜索
    返回: start_idx, end_idx, duration_us, teo_smooth, threshold
    """
    # 1. 计算 Teager 能量
    teo = calculate_teager_energy(signal)
    
    # 2. 10点滑动平均
    teo_smooth = moving_average(teo, window_size=10)
    
    # 3. 噪声基准：基于全局 Teager 能量的第 40 百分位数
    noise_floor = np.percentile(teo_smooth, 40)
    threshold = noise_floor * 3
    
    # --- 修改部分：定位能量最大值点 ---
    # 找到 Teager 能量曲线中的峰值索引
    max_idx = np.argmax(teo_smooth)
    
    # 如果最大值本身就低于阈值，说明可能没有有效信号
    if teo_smooth[max_idx] < threshold:
        # 返回一个零长度或默认值，避免程序崩溃
        return max_idx, max_idx, 0.0, teo_smooth, threshold

    # 4. 从最大值点 (max_idx) 向前后搜索
    
    # 向左（前）搜索起点
    start_idx = max_idx
    while start_idx > 0 and teo_smooth[start_idx] > threshold:
        start_idx -= 1
        
    # 向右（后）搜索终点
    end_idx = max_idx
    while end_idx < len(teo_smooth) - 1 and teo_smooth[end_idx] > threshold:
        end_idx += 1
        
    # 计算时长 (微秒)
    # 计算公式：(样本数 / 采样率) * 1,000,000
    duration_us = (end_idx - start_idx) / fs * 1e6
    
    return start_idx, end_idx, duration_us, teo_smooth, threshold

In [42]:
import random

# 1. 获取并随机采样
all_wav_files = glob.glob(os.path.join(data_dir, '**', 'Pulse_*.wav'), recursive=True)
num_to_sample = min(2000, len(all_wav_files))
selected_files = random.sample(all_wav_files, num_to_sample)

# 2. 定义记录文件路径
log_file_path = os.path.join(output_base, "duration_results.txt")

print(f"正在处理 {num_to_sample} 个文件并将时长数据保存至 {log_file_path}...")

# 3. 执行计算逻辑
results_list = [] # 用于传递给下一个 Cell，同时也写入文件备份

with open(log_file_path, "w", encoding="utf-8") as f:
    # 写入表头
    f.write("file_path|duration_us|start_idx|end_idx\n")
    
    for file_path in tqdm(selected_files):
        try:
            fs, data = wav.read(file_path)
            if len(data.shape) > 1: data = data[:, 0]
            data_norm = data.astype(float) / np.max(np.abs(data)) if np.max(np.abs(data)) != 0 else data.astype(float)
            
            # 计算边界和时长
            start_i, end_i, dur, teo_s, thresh = detect_boundaries(data_norm, fs)
            
            # 记录数据 (文件路径 | 时长 | 起始点 | 结束点)
            line = f"{file_path}|{dur:.2f}|{start_i}|{end_i}"
            f.write(line + "\n")
            
            # 存入列表供 Cell 4 直接使用，避免重新读取 txt
            results_list.append({
                'path': file_path,
                'dur': dur,
                'start_i': start_i,
                'end_i': end_i
            })
            
        except Exception as e:
            print(f"解析 {file_path} 出错: {e}")

print(f"时长计算完成，结果已保存至: {log_file_path}")

正在处理 2000 个文件并将时长数据保存至 D:\Project_Github\Indo-Pacific-humpback-dolphin\04_Distinguish\Output_Classification\duration_results.txt...


100%|██████████| 2000/2000 [00:00<00:00, 3498.63it/s]

时长计算完成，结果已保存至: D:\Project_Github\Indo-Pacific-humpback-dolphin\04_Distinguish\Output_Classification\duration_results.txt


In [43]:
import scipy.io.wavfile as wav
import numpy as np

# 计数器
stats = {'Clean': 0, 'Moderate': 0, 'Severe': 0}

print("开始处理：修改信号尾部 -> 分类 -> 保存图片与音频...")

for item in tqdm(results_list):
    file_path = item['path']
    dur = item['dur']
    start_i = item['start_i']
    end_i = item['end_i']
    
    # 1. 读取原始数据
    fs, data = wav.read(file_path)
    # 确保是浮点数进行处理，防止溢出或截断
    modified_data = data.copy().astype(float)
    
    # --- 核心逻辑：将结束点后的幅值设置为和结束点一致 ---
    if end_i < len(modified_data):
        if len(modified_data.shape) > 1: # 如果是多声道
            val_at_end = modified_data[end_i, :]
            modified_data[end_i:, :] = val_at_end
        else: # 单声道
            val_at_end = modified_data[end_i]
            modified_data[end_i:] = val_at_end
            
    # 2. 判定混响严重程度
    if dur < 50:
        cat = 'Clean'
    elif 50 <= dur < 100:
        cat = 'Moderate'
    else:
        cat = 'Severe'
    
    stats[cat] += 1

    # 3. 准备绘图用的归一化数据 (使用修改后的数据)
    if len(modified_data.shape) > 1: 
        plot_data = modified_data[:, 0]
    else:
        plot_data = modified_data
    
    max_val = np.max(np.abs(plot_data))
    data_norm = plot_data / max_val if max_val != 0 else plot_data
    
    # 构造时间轴 (0-200us)
    time_axis = np.linspace(0, (len(data_norm) / fs) * 1e6, len(data_norm))
    start_time = (start_i / fs) * 1e6
    end_time = (end_i / fs) * 1e6

    # 重新计算 Teager 能量曲线用于展示
    teo = calculate_teager_energy(data_norm)
    teo_s = moving_average(teo, 10)
    noise_floor = np.percentile(teo_s, 40)
    thresh = noise_floor * 3

    # 4. 准备保存路径
    parent_folder = os.path.basename(os.path.dirname(file_path))
    file_name_base = os.path.basename(file_path).replace('.wav', '')
    target_dir = os.path.join(output_base, cat)
    
    save_name_png = f"{parent_folder}_{file_name_base}.png"
    save_name_wav = f"{parent_folder}_{file_name_base}.wav"

    # --- 保存修改后的音频文件 ---
    # 转回原始数据类型（通常是 int16）以保持兼容性
    final_wav_data = modified_data.astype(data.dtype)
    wav.write(os.path.join(target_dir, save_name_wav), fs, final_wav_data)

    # --- 绘图与保存图片 ---
    plt.figure(figsize=(10, 6))
    
    # 子图1: 修改后的波形
    plt.subplot(2, 1, 1)
    plt.plot(time_axis, data_norm, color='gray', alpha=0.6, label='Modified Signal')
    plt.axvspan(start_time, end_time, color='red', alpha=0.2, label=f'Click Region ({dur:.1f} us)')
    # 画一条虚线标记处理开始的位置
    plt.axvline(x=end_time, color='blue', linestyle='--', alpha=0.5, label='Tail Processed')
    
    plt.title(f"Class: {cat} | Dur: {dur:.1f}us | File: {parent_folder}/{file_name_base}")
    plt.xlabel('Time (us)')
    plt.ylabel('Normalized Amp')
    plt.xlim(0, 200)
    plt.legend(loc='upper right')
    
    # 子图2: Teager 能量曲线
    plt.subplot(2, 1, 2)
    plt.plot(time_axis, teo_s, color='blue')
    plt.axhline(y=thresh, color='green', linestyle='--', label='Threshold')
    plt.axvline(x=start_time, color='red', linestyle=':')
    plt.axvline(x=end_time, color='red', linestyle=':')
    plt.xlabel('Time (us)')
    plt.ylabel('Teager Energy')
    plt.xlim(0, 200)
    
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, save_name_png))
    plt.close()

# 5. 输出统计
print("\n" + "="*30)
print("处理完成！尾部信号已平坦化。")
for k, v in stats.items():
    print(f"- {k}: {v} 个文件")
print(f"输出位置: {output_base}")
print("="*30)

开始处理：修改信号尾部 -> 分类 -> 保存图片与音频...


100%|██████████| 2000/2000 [05:14<00:00,  6.36it/s]


处理完成！尾部信号已平坦化。
- Clean: 131 个文件
- Moderate: 753 个文件
- Severe: 1116 个文件
输出位置: D:\Project_Github\Indo-Pacific-humpback-dolphin\04_Distinguish\Output_Classification
